# Проект 2. Прогноз цен на жильё в Бостоне с помощью линейной регрессии

Нужно предсказать медианную стоимость дома в районе Бостона (`MEDV`, в тысячах долларов) по характеристикам района.

Данные: Boston Housing, 506 районов и 13 признаков. Например, `RM` — среднее число комнат, `CRIM` — уровень преступности, `LSTAT` — доля населения с низким доходом, `PTRATIO` — сколько учеников приходится на одного учителя. Загружаю из пакета `MLDatasets`.

Метод: обычная множественная линейная регрессия из пакета `GLM`, цена считается как взвешенная сумма всех признаков плюс константа.

Данные перемешиваю и делю 80/20. Качество смотрю по R² на обучающей выборке и по ошибкам MSE и MAE на тестовой.

In [ ]:
import Pkg
Pkg.add(["DataFrames", "CSV", "GLM", "StatsBase", "MLUtils", "MLBase", "MLDatasets", "Random"])

In [ ]:
using GLM, StatsBase, DataFrames, MLUtils, MLDatasets, Random

# Загружаем датасет
data = MLDatasets.BostonHousing().dataframe
println("Размер датасета: ", size(data))  # (506, 14)

# Делим на train/test (80/20)
Random.seed!(42)
train_idx, test_idx = splitobs(shuffle(1:nrow(data)), at=0.8)
train = data[train_idx, :]
test  = data[test_idx, :]
println("Train: ", nrow(train), " | Test: ", nrow(test))

# Линейная регрессию
fm = @formula(MEDV ~ CRIM + ZN + INDUS + CHAS + NOX + RM + AGE + DIS +
                     RAD + TAX + PTRATIO + B + LSTAT)
linear_regressor = lm(fm, train)

# Оценка на обучающей выборке
r2_train = r2(linear_regressor)
println("R² на train: ", round(r2_train, digits=4))

# Предсказание и MSE на тестовой выборке
predictions = predict(linear_regressor, test)
mse = mean((predictions .- test.MEDV) .^ 2)
println("MSE на test: ", round(mse, digits=2))

# Дополнительно MAE
mae = mean(abs.(predictions .- test.MEDV))
println("MAE на test: ", round(mae, digits=2))

Размер датасета: (506, 14)
Train: 405 | Test: 101
R² на train: 0.7378
MSE на test: 19.96
MAE на test: 3.27


## Вывод

R² на обучающей выборке 0.74, то есть модель объясняет примерно три четверти разброса цен. На тесте MSE = 19.96, MAE = 3.27: в среднем прогноз отличается от реальной цены примерно на 3.3 тыс. долларов при средней цене около 22 тыс.

Сначала данные делились без перемешивания, и MSE получался 33.3. Оказалось, что строки в датасете упорядочены по районам, и в тест попадали только последние районы, которые заметно отличаются от остальных. После перемешивания ошибка упала почти вдвое.

Линейная регрессия удобна тем, что по коэффициентам видно, как каждый признак влияет на цену. Но не все зависимости здесь линейные (например, цена от `LSTAT`), поэтому более сложные модели, скорее всего, дали бы меньшую ошибку.